# Grafuri si algoritmi pentru grafuri: BFS, DFS si Dijkstra


# 1. Notiuni de baza despre grafuri

Un **graf** este o structura folosita pentru a reprezenta relatii intre obiecte.

Un graf este format din:
- **noduri** sau **varfuri**;
- **muchii** sau **arce**.

Exemple:
- orase conectate prin drumuri;
- persoane conectate intr-o retea sociala;
- statii de metrou conectate prin linii;
- calculatoare conectate intr-o retea;
- pagini web conectate prin linkuri.

## Tipuri de grafuri

### Graf neorientat
Muchiile nu au directie.  
Daca exista o muchie intre `A` si `B`, se poate merge si de la `A` la `B`, si de la `B` la `A`.

### Graf orientat
Muchiile au directie.  
Daca exista `A -> B`, nu inseamna automat ca exista si `B -> A`.

### Graf ponderat
Fiecare muchie are un cost:
- distanta;
- timp;
- pret;
- risc;
- consum de energie.

Algoritmul Dijkstra se foloseste pentru grafuri ponderate cu costuri nenegative.


# 2. Cum modelam o problema cu grafuri

Inainte de cod, trebuie stabilit ce reprezinta nodurile si muchiile.

## Intrebari utile

1. **Ce sunt nodurile?**  
   Orase, persoane, servere, statii, pagini, task-uri, celule dintr-o matrice.

2. **Ce sunt muchiile?**  
   Drumuri, conexiuni, relatii, dependente, mutari posibile.

3. **Graful este orientat sau neorientat?**  
   Daca relatia este reciproca, graful este neorientat.  
   Daca relatia are sens, graful este orientat.

4. **Muchiile au costuri?**  
   Daca da, avem graf ponderat.  
   Daca nu, putem considera fiecare muchie cu cost 1.

5. **Ce vrem sa calculam?**  
   - parcurgere;
   - existenta unui drum;
   - drum minim;
   - toate nodurile accesibile;
   - componente conexe.


# 3. Reprezentarea grafurilor in Python

Cea mai folosita reprezentare este **lista de adiacenta**.

Pentru fiecare nod, retinem lista vecinilor sai.

Exemplu:
```text
A -> B, C
B -> D
C -> D
D -> nimic
```


In [ ]:
graf = {
    "A": ["B", "C"],
    "B": ["D"],
    "C": ["D"],
    "D": []
}

print(graf)

## Reprezentare pentru graf ponderat

Pentru grafuri ponderate, fiecare vecin este retinut impreuna cu un cost.

Exemplu:
```text
A -> B cost 4
A -> C cost 2
```


In [ ]:
graf_ponderat = {
    "A": [("B", 4), ("C", 2)],
    "B": [("D", 5)],
    "C": [("B", 1), ("D", 8)],
    "D": []
}

print(graf_ponderat)

# 4. BFS – Breadth-First Search

BFS este o parcurgere pe niveluri.

Ideea:
- pornim dintr-un nod;
- vizitam mai intai vecinii directi;
- apoi vecinii vecinilor;
- continuam pe niveluri.

BFS foloseste o **coada**.

Este potrivit pentru:
- grafuri neponderate;
- distante minime ca numar de muchii;
- verificarea accesibilitatii.


In [ ]:
from collections import deque

def bfs(graf, start):
    coada = deque([start])
    vizitat = set([start])
    ordine = []

    while coada:
        nod = coada.popleft()
        ordine.append(nod)

        for vecin in graf[nod]:
            if vecin not in vizitat:
                vizitat.add(vecin)
                coada.append(vecin)

    return ordine

print(bfs(graf, "A"))

## Distante cu BFS in graf neponderat

Daca toate muchiile au acelasi cost, BFS calculeaza distante minime in numar de muchii.


In [ ]:
def bfs_distances(graf, start):
    distante = {nod: float("inf") for nod in graf}
    distante[start] = 0
    coada = deque([start])

    while coada:
        nod = coada.popleft()

        for vecin in graf[nod]:
            if distante[vecin] == float("inf"):
                distante[vecin] = distante[nod] + 1
                coada.append(vecin)

    return distante

print(bfs_distances(graf, "A"))

# 5. DFS – Depth-First Search

DFS este o parcurgere in adancime.

Ideea:
- pornim dintr-un nod;
- alegem un vecin;
- mergem cat mai adanc;
- revenim cand nu mai putem continua.

DFS poate fi scris recursiv sau iterativ.


In [ ]:
def dfs_recursiv(graf, nod, vizitat=None, ordine=None):
    if vizitat is None:
        vizitat = set()
    if ordine is None:
        ordine = []

    vizitat.add(nod)
    ordine.append(nod)

    for vecin in graf[nod]:
        if vecin not in vizitat:
            dfs_recursiv(graf, vecin, vizitat, ordine)

    return ordine

print(dfs_recursiv(graf, "A"))

# 6. Problema drumului minim

Problema drumului minim cere sa gasim drumul cu cost total minim intre noduri.

Daca graful este neponderat, BFS este suficient.

Daca graful este ponderat cu costuri nenegative, folosim **Dijkstra**.

## Conditie esentiala
Dijkstra functioneaza corect doar daca toate costurile muchiilor sunt:
```text
>= 0
```

Daca exista muchii negative, se folosesc alti algoritmi, cum ar fi Bellman-Ford.


# 7. Algoritmul Dijkstra – idee intuitiva

Dijkstra calculeaza distanta minima de la un nod sursa la toate celelalte noduri.

Algoritmul pastreaza pentru fiecare nod:
- cea mai buna distanta cunoscuta pana acum;
- eventual parintele prin care ajungem la acel nod.

La fiecare pas:
1. alegem nodul cu cea mai mica distanta provizorie;
2. il consideram finalizat;
3. incercam sa imbunatatim distantele vecinilor sai.

Aceasta imbunatatire se numeste **relaxare**.


## Relaxarea unei muchii

Pentru o muchie:
```text
u -> v
```

cu cost `w`, verificam:

```text
dist[v] > dist[u] + w
```

Daca aceasta conditie este adevarata, inseamna ca am gasit un drum mai bun catre `v`.

Atunci actualizam:

```text
dist[v] = dist[u] + w
```


# 8. Exemplu Dijkstra pas cu pas

Graf:
```text
A --4--> B
A --2--> C
C --1--> B
B --5--> D
C --8--> D
```

Initial:
```text
dist[A] = 0
dist[B] = infinit
dist[C] = infinit
dist[D] = infinit
```

## Pasul 1
Alegem `A`.

Relaxam:
- A -> B: B devine 4
- A -> C: C devine 2

```text
A: 0
B: 4
C: 2
D: inf
```

## Pasul 2
Alegem `C`, pentru ca are distanta minima 2.

Relaxam:
- C -> B: 2 + 1 = 3, deci B se imbunatateste de la 4 la 3
- C -> D: 2 + 8 = 10, deci D devine 10

```text
A: 0
B: 3
C: 2
D: 10
```

## Pasul 3
Alegem `B`.

Relaxam:
- B -> D: 3 + 5 = 8, deci D se imbunatateste de la 10 la 8

```text
A: 0
B: 3
C: 2
D: 8
```

Rezultatul final:
```text
A: 0
C: 2
B: 3
D: 8
```


# 9. Implementare Dijkstra cu heap

Folosim `heapq`, o coada de prioritati.

Aceasta permite extragerea rapida a nodului cu cea mai mica distanta curenta.


In [ ]:
import heapq

def dijkstra(graf, start):
    distante = {nod: float("inf") for nod in graf}
    distante[start] = 0

    heap = [(0, start)]

    while heap:
        dist_curenta, nod_curent = heapq.heappop(heap)

        if dist_curenta > distante[nod_curent]:
            continue

        for vecin, cost in graf[nod_curent]:
            dist_noua = dist_curenta + cost

            if dist_noua < distante[vecin]:
                distante[vecin] = dist_noua
                heapq.heappush(heap, (dist_noua, vecin))

    return distante

print(dijkstra(graf_ponderat, "A"))

# 10. Dijkstra cu afisarea pasilor

Aceasta versiune este utila pentru intelegere.  
Afiseaza nodul extras si distantele actualizate.


In [ ]:
def dijkstra_verbose(graf, start):
    distante = {nod: float("inf") for nod in graf}
    distante[start] = 0
    heap = [(0, start)]
    pas = 1

    while heap:
        dist_curenta, nod_curent = heapq.heappop(heap)

        if dist_curenta > distante[nod_curent]:
            continue

        print(f"Pasul {pas}: extragem {nod_curent}, distanta = {dist_curenta}")
        pas += 1

        for vecin, cost in graf[nod_curent]:
            dist_noua = dist_curenta + cost

            if dist_noua < distante[vecin]:
                print(f"  Relaxam {nod_curent}->{vecin}: {distante[vecin]} devine {dist_noua}")
                distante[vecin] = dist_noua
                heapq.heappush(heap, (dist_noua, vecin))

        print("  Distante:", distante)
        print()

    return distante

dijkstra_verbose(graf_ponderat, "A")

# 11. Reconstructia drumului minim

Dijkstra poate calcula nu doar costul minim, ci si drumul efectiv.

Pentru asta memoram pentru fiecare nod un `parinte`.

Daca am ajuns optim la `D` prin `B`, atunci:
```text
parinte[D] = B
```

La final, reconstruim drumul mergand inapoi de la destinatie la sursa.


In [ ]:
def dijkstra_with_path(graf, start):
    distante = {nod: float("inf") for nod in graf}
    parinte = {nod: None for nod in graf}

    distante[start] = 0
    heap = [(0, start)]

    while heap:
        dist_curenta, nod_curent = heapq.heappop(heap)

        if dist_curenta > distante[nod_curent]:
            continue

        for vecin, cost in graf[nod_curent]:
            dist_noua = dist_curenta + cost

            if dist_noua < distante[vecin]:
                distante[vecin] = dist_noua
                parinte[vecin] = nod_curent
                heapq.heappush(heap, (dist_noua, vecin))

    return distante, parinte

def reconstruieste_drum(parinte, start, destinatie):
    drum = []
    nod = destinatie

    while nod is not None:
        drum.append(nod)
        nod = parinte[nod]

    drum.reverse()

    if drum and drum[0] == start:
        return drum

    return []

dist, par = dijkstra_with_path(graf_ponderat, "A")
print(dist)
print(reconstruieste_drum(par, "A", "D"))

# 12. Drum minim catre o singura destinatie

Daca ne intereseaza doar un nod destinatie, putem opri algoritmul cand acel nod este extras din heap.

In acel moment, distanta lui este finala.


In [ ]:
def dijkstra_to_target(graf, start, target):
    distante = {nod: float("inf") for nod in graf}
    parinte = {nod: None for nod in graf}

    distante[start] = 0
    heap = [(0, start)]

    while heap:
        dist_curenta, nod_curent = heapq.heappop(heap)

        if dist_curenta > distante[nod_curent]:
            continue

        if nod_curent == target:
            break

        for vecin, cost in graf[nod_curent]:
            dist_noua = dist_curenta + cost

            if dist_noua < distante[vecin]:
                distante[vecin] = dist_noua
                parinte[vecin] = nod_curent
                heapq.heappush(heap, (dist_noua, vecin))

    return distante[target], reconstruieste_drum(parinte, start, target)

print(dijkstra_to_target(graf_ponderat, "A", "D"))

# 13. Exemplu practic: harta cu orase

Nodurile sunt orase.  
Muchiile sunt drumuri.  
Costurile sunt distante.


In [ ]:
harta = {
    "Bucuresti": [("Ploiesti", 70), ("Pitesti", 120)],
    "Ploiesti": [("Bucuresti", 70), ("Brasov", 160)],
    "Pitesti": [("Bucuresti", 120), ("Sibiu", 140)],
    "Brasov": [("Ploiesti", 160), ("Sibiu", 150)],
    "Sibiu": [("Pitesti", 140), ("Brasov", 150), ("Cluj", 270)],
    "Cluj": [("Sibiu", 270)]
}

dist, par = dijkstra_with_path(harta, "Bucuresti")

for oras in harta:
    print("Bucuresti ->", oras, "cost =", dist[oras], "drum =", reconstruieste_drum(par, "Bucuresti", oras))

# 14. BFS versus Dijkstra

BFS si Dijkstra pot parea asemanatoare, dar rezolva probleme diferite.

## BFS
Folosim BFS cand:
- graful este neponderat;
- toate muchiile au acelasi cost.

## Dijkstra
Folosim Dijkstra cand:
- graful este ponderat;
- costurile sunt diferite;
- costurile sunt nenegative.

## Exemplu unde BFS nu este suficient

```text
A -> B cost 100
A -> C cost 1
C -> B cost 1
```

BFS ar prefera `A -> B`, pentru ca are o singura muchie.  
Dijkstra alege `A -> C -> B`, pentru ca are cost 2.


In [ ]:
graf_diferenta = {
    "A": [("B", 100), ("C", 1)],
    "B": [],
    "C": [("B", 1)]
}

dist, par = dijkstra_with_path(graf_diferenta, "A")
print("Cost A -> B:", dist["B"])
print("Drum:", reconstruieste_drum(par, "A", "B"))

# 15. Noduri inaccesibile

Daca un nod ramane cu distanta `inf`, inseamna ca nu exista drum de la sursa la acel nod.


In [ ]:
graf_inaccesibil = {
    "A": [("B", 2)],
    "B": [("C", 3)],
    "C": [],
    "D": [("E", 1)],
    "E": []
}

dist = dijkstra(graf_inaccesibil, "A")
print(dist)

for nod, d in dist.items():
    if d == float("inf"):
        print(nod, "nu este accesibil din A")

# 16. Construirea unui graf neorientat ponderat

Pentru graf neorientat, fiecare muchie trebuie introdusa in ambele sensuri.


In [ ]:
def build_undirected_weighted_graph(edges):
    graf = {}

    for u, v, cost in edges:
        if u not in graf:
            graf[u] = []
        if v not in graf:
            graf[v] = []

        graf[u].append((v, cost))
        graf[v].append((u, cost))

    return graf

edges = [
    ("A", "B", 4),
    ("A", "C", 2),
    ("C", "B", 1),
    ("B", "D", 5),
    ("C", "D", 8)
]

g_neorientat = build_undirected_weighted_graph(edges)
print(g_neorientat)
print(dijkstra(g_neorientat, "A"))

# 17. Verificarea costurilor negative

Dijkstra nu trebuie aplicat pe grafuri cu muchii negative.

Putem scrie o functie care verifica acest lucru.


In [ ]:
def has_negative_edges(graf):
    for nod in graf:
        for vecin, cost in graf[nod]:
            if cost < 0:
                return True
    return False

def dijkstra_safe(graf, start):
    if has_negative_edges(graf):
        raise ValueError("Dijkstra nu se aplica pe grafuri cu muchii negative.")
    return dijkstra(graf, start)

g_negativ = {
    "A": [("B", 2), ("C", 5)],
    "B": [],
    "C": [("B", -10)]
}

print(has_negative_edges(graf_ponderat))
print(has_negative_edges(g_negativ))

try:
    print(dijkstra_safe(g_negativ, "A"))
except ValueError as e:
    print("Eroare:", e)

# 18. Aplicatie practica: retea de servere

Nodurile pot fi servere.  
Muchiile pot fi conexiuni.  
Costurile pot fi latente.


In [ ]:
retea = {
    "Server1": [("Server2", 10), ("Server3", 3)],
    "Server2": [("Server4", 2)],
    "Server3": [("Server2", 1), ("Server4", 8), ("Server5", 2)],
    "Server4": [("Server5", 4)],
    "Server5": []
}

dist, par = dijkstra_with_path(retea, "Server1")
print("Cost minim catre Server5:", dist["Server5"])
print("Ruta:", reconstruieste_drum(par, "Server1", "Server5"))

# 19. Complexitatea algoritmului Dijkstra

Cu heap:
```text
O((V + E) log V)
```

unde:
- `V` = numarul de noduri;
- `E` = numarul de muchii.

Fara heap, o implementare simpla poate ajunge la:
```text
O(V^2)
```

Dijkstra este eficient pentru grafuri:
- ponderate;
- cu costuri nenegative;
- reprezentate prin liste de adiacenta.


# 20. Erori frecvente

1. Aplicarea lui Dijkstra pe grafuri cu costuri negative.
2. Neinitializarea distantelor cu infinit.
3. Uitarea actualizarii parintelui.
4. Confuzia dintre BFS si Dijkstra.
5. Neintroducerea muchiilor in ambele sensuri pentru grafuri neorientate.
6. Folosirea lui Dijkstra cand BFS ar fi suficient.


# 21. Exercitii rezolvate

## Exercitiul 1
Calculati distantele minime din `A` pentru graful:
```text
A -> B (3)
A -> C (1)
C -> B (1)
B -> D (4)
C -> D (7)
```


In [ ]:
g = {
    "A": [("B", 3), ("C", 1)],
    "B": [("D", 4)],
    "C": [("B", 1), ("D", 7)],
    "D": []
}

print(dijkstra(g, "A"))

## Exercitiul 2
Reconstituiti drumul minim de la `A` la `D`.


In [ ]:
dist, par = dijkstra_with_path(g, "A")
print("Distanta:", dist["D"])
print("Drum:", reconstruieste_drum(par, "A", "D"))

# 22. Exercitii propuse

## Nivel introductiv
1. Reprezentati un graf neorientat cu 5 noduri.
2. Implementati BFS pentru graful respectiv.
3. Implementati DFS pentru graful respectiv.
4. Transformati un graf neponderat intr-un graf ponderat cu toate costurile egale cu 1.


# 23. Rezumat

## BFS
- pentru grafuri neponderate;
- gaseste drumul minim ca numar de muchii;
- foloseste coada.

## DFS
- exploreaza in adancime;
- este util pentru conectivitate si componente;
- foloseste recursivitate sau stiva.

## Dijkstra
- pentru grafuri ponderate cu costuri nenegative;
- calculeaza costuri minime;
- foloseste relaxarea muchiilor;
- foloseste eficient coada de prioritati.

## De retinut
Daca toate muchiile au acelasi cost, BFS este suficient.  
Daca muchiile au costuri diferite, dar nenegative, folosim Dijkstra.
